# Case Study 1: Algorithmic Auditing of Systemic Contradictions
**Objective:** Leverage the ESC Knowledge Graph (ESC-KG) to investigate the topological correlation between the two primary monitoring mechanisms of the European Social Charter: the periodic reporting system and the optional collective complaints procedure.

Traditional text-based retrieval methods fall short in capturing cross-procedural dynamics due to the highly fragmented nature of the underlying legal corpus. To overcome this limitation, this analysis utilizes graph traversal to partition State-Article pairs into two distinct topological cohorts. The first cohort consists of pairs that exist solely within the boundaries of the reporting system. The second cohort isolates pairs that intersect with the collective complaints mechanism, having been targeted by at least one complaint during their procedural lifecycle.

By programmatically extracting the absolute counts of categorical compliance verdicts (Conformity, Non-Conformity, and Deferred) for both cohorts, we transform the decentralized legal network into a structured contingency table. Ultimately, we apply a Chi-Square test of independence to ascertain whether the topological presence of a collective complaint statistically correlates with a divergent distribution of reporting verdicts, thereby mathematically validating the friction points within the European Social Charter ecosystem.


In [23]:
import json
import networkx as nx
import pandas as pd
from collections import defaultdict


# 1. Load the exported Knowledge Graph
print("Loading ESC Knowledge Graph...")
with open('graph.json', 'r') as f:
    data = json.load(f)

G = nx.node_link_graph(data)

print(f"Graph loaded successfully: {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

Loading ESC Knowledge Graph...
Graph loaded successfully: 29137 nodes and 30208 edges.


In [24]:
from datasets import load_dataset
dataset = load_dataset("rmignone/HUDOC-ESC", data_files="ESC_Corpus_Final.parquet")

node_df = dataset['train'].to_pandas()

In [25]:
# 2. Graph Traversal: Mapping Documents to Articles by State
# We will create dictionaries mapping (State, Article) to their respective procedural documents.

reporting_docs_map = defaultdict(list)   # Stores 'Conclusion' nodes (as doc_ids)
complaints_map = defaultdict(list)  # Stores 'ECSR Finding' or 'Merit' nodes

# Target document types based on our schema
REPORTING_TYPES = ["Conclusion"]
COMPLAINT_TYPES = ["ECSR Finding", "Merit", "Case Merit"]

print("Traversing edges to map state-article relationships...")

for u, v, edge_data in G.edges(data=True):
    rel_type = edge_data.get("relationship_type", "")

    # We look for edges connecting legal documents (u) to Charter Articles (v)
    if rel_type in ["assesses_compliance_of", "interpreted_by", "invoked_by"]:
        doc_node = G.nodes[u]
        article_id = v

        doc_types = doc_node.get("document_type_text", [])
        state = doc_node.get("state_party", None)

        # Skip if state is missing or NaN
        if not state or pd.isna(state):
            continue

        # Ensure state is parsed cleanly (handling semicolon-separated strings if any)
        states = [s.strip() for s in str(state).split(";") if s.strip()]

        for s in states:
            target_key = (s, article_id)

            # Categorize into Reporting System
            if any(dt in REPORTING_TYPES for dt in doc_types):
                reporting_docs_map[target_key].append(u)

            # Categorize into Collective Complaints
            elif any(dt in COMPLAINT_TYPES for dt in doc_types):
                complaints_map[target_key].append(u)

print(f"Found {len(reporting_docs_map)} unique (State, Article) pairs in Reporting System (doc_ids).")
print(f"Found {len(complaints_map)} unique (State, Article) pairs in Collective Complaints (doc_ids).")

Traversing edges to map state-article relationships...
Found 3174 unique (State, Article) pairs in Reporting System (doc_ids).
Found 165 unique (State, Article) pairs in Collective Complaints (doc_ids).


In [26]:
# 3. Topological Correlation Analysis: Reporting vs Complaints
# We will check if the presence of a complaint correlates with more 'NC' (Non-Conformity) in reports.

reporting_map = defaultdict(list)
complaints_set = set()

# Ensure node_df is indexed by 'base_id' or 'id'
# node_df = node_df.set_index('id')

def get_conclusion_verdict(node_id):
    """Retrieves the verdict strictly for Conclusion nodes."""
    clean_id = str(node_id).strip()
    try:
        verdict = node_df[node_df['base_id'] == clean_id]['decision_type']
        if isinstance(verdict, pd.Series):
            verdict = verdict.iloc[0]
        decision = str(verdict).strip().upper()
        if decision in ["C", "NC", "D"]:
            return decision
    except KeyError:
        pass
    return "Unknown"

print("Traversing graph to map State-Article pairs...")

for u, v, edge_data in G.edges(data=True):
    rel_type = edge_data.get("relationship_type", "")

    doc_id = None
    article_id = None

    # Handle the specific direction of your edges
    if rel_type == "assesses_compliance_of":
        # Document -> Article
        doc_id = u
        article_id = v
    elif rel_type == "invoked_by":
        # Article -> Document (Complaint)
        doc_id = v
        article_id = u

    if doc_id and article_id:
        doc_node = G.nodes[doc_id]
        doc_types = doc_node.get("document_type_text", [])
        state = doc_node.get("state_party", None)

        if not state or pd.isna(state):
            continue

        # Handle cases where multiple states are grouped in one string
        states = [s.strip() for s in str(state).split(";") if s.strip()]

        for s in states:
            target_key = (s, article_id)

            # Map Reporting Outcomes
           # Map Reporting Outcomes
            if "Conclusion" in doc_types:
                verdict = get_conclusion_verdict(doc_id)
                if verdict != "Unknown":
                    # Store as a tuple: (Verdict, Document ID)
                    reporting_map[target_key].append((verdict, doc_id))

            # Track if this (State, Article) ever faced a complaint/merit
            elif any(dt in ["Complaint", "Case Merit", "Case Admissibility"] for dt in doc_types):
                complaints_set.add(target_key)

print(f"Mapped verdicts for {len(reporting_map)} unique (State, Article) pairs.")
print(f"Identified {len(complaints_set)} (State, Article) pairs targeted by collective complaints.")

Traversing graph to map State-Article pairs...
Mapped verdicts for 3174 unique (State, Article) pairs.
Identified 414 (State, Article) pairs targeted by collective complaints.


In [27]:
# 4. Analyze and Compare the Distributions
# We separate the reporting verdicts into two cohorts based on network topology

targeted_verdicts = []
untargeted_verdicts = []

for target_key, tuple_list in reporting_map.items():
    # Extract just the verdict string [v[0]] from the tuple
    just_verdicts = [v[0] for v in tuple_list]

    if target_key in complaints_set:
        targeted_verdicts.extend(just_verdicts)
    else:
        untargeted_verdicts.extend(just_verdicts)

# ... (The rest of Cell 4 remains exactly the same)

# Convert to Pandas for easy counting and percentage calculation
targeted_df = pd.Series(targeted_verdicts).value_counts(normalize=True) * 100
untargeted_df = pd.Series(untargeted_verdicts).value_counts(normalize=True) * 100

comparison_df = pd.DataFrame({
    "Never Faced a Complaint (%)": untargeted_df,
    "Faced at least one Complaint (%)": targeted_df
}).fillna(0).round(2)

print("\n--- TOPOLOGICAL CORRELATION RESULTS ---")
print("Distribution of Periodic Reporting Verdicts (C=Conformity, NC=Non-Conformity, D=Deferred):")
display(comparison_df)

# Export for the paper
comparison_df.to_csv("topological_correlation_complaints_vs_reports.csv")


--- TOPOLOGICAL CORRELATION RESULTS ---
Distribution of Periodic Reporting Verdicts (C=Conformity, NC=Non-Conformity, D=Deferred):


,Never Faced a Complaint (%),Faced at least one Complaint (%)
C,56.49,54.06
NC,23.39,25.21
D,20.12,20.73


In [28]:
from scipy.stats import chi2_contingency

# 1. Get the absolute counts for the contingency table
targeted_counts = pd.Series(targeted_verdicts).value_counts()
untargeted_counts = pd.Series(untargeted_verdicts).value_counts()

# 2. Build the contingency table
contingency_table = pd.DataFrame({
    "Faced Complaint": targeted_counts,
    "Never Faced Complaint": untargeted_counts
}).fillna(0) # Fill NaN with 0 for categories that might be missing

print("--- CONTINGENCY TABLE (ABSOLUTE COUNTS) ---")
display(contingency_table)

# 3. Perform the Chi-Square Test
chi2_stat, p_value, dof, expected = chi2_contingency(contingency_table)

print("\n--- STATISTICAL SIGNIFICANCE TEST ---")
print(f"Chi-Square Statistic: {chi2_stat:.4f}")
print(f"P-value: {p_value:.4e}")
print(f"Degrees of Freedom: {dof}")

# 4. Interpretation
alpha = 0.05
if p_value < alpha:
    print("\nVerdict: SIGNIFICANT (p < 0.05).")
    print("We reject the null hypothesis. The distribution of reporting verdicts for articles targeted by complaints is statistically different from those that are not.")
else:
    print("\nVerdict: NOT SIGNIFICANT (p >= 0.05).")
    print("We cannot reject the null hypothesis. There is no statistically significant difference between the two groups.")

--- CONTINGENCY TABLE (ABSOLUTE COUNTS) ---


,Faced Complaint,Never Faced Complaint
C,1844,13096
NC,860,5422
D,707,4665



--- STATISTICAL SIGNIFICANCE TEST ---
Chi-Square Statistic: 7.8529
P-value: 1.9714e-02
Degrees of Freedom: 2

Verdict: SIGNIFICANT (p < 0.05).
We reject the null hypothesis. The distribution of reporting verdicts for articles targeted by complaints is statistically different from those that are not.


# Nuovo

In [32]:
# 5. Calculate Effect Size (Cramer's V)
import numpy as np

# Total number of observations
n = contingency_table.sum().sum()

# Minimum dimension minus 1 for Cramer's V formula
# shape is (3 rows, 2 columns), so min_dim = min(3-1, 2-1) = 1
min_dim = min(contingency_table.shape) - 1

cramer_v = np.sqrt(chi2_stat / (n * min_dim))

print("--- EFFECT SIZE ---")
print(f"Cramer's V: {cramer_v:.4f}")

# Standard interpretation guidelines
if cramer_v < 0.1:
    print("Interpretation: Small or weak effect.")
elif cramer_v < 0.3:
    print("Interpretation: Moderate effect.")
else:
    print("Interpretation: Strong effect.")

--- EFFECT SIZE ---
Cramer's V: 0.0172
Interpretation: Small or weak effect.


In [30]:
import datetime

temporal_targeted_verdicts = []
failed_temporal_matches = 0
malformed_reporting_entries = 0 # Counter for malformed entries in reporting_map

print("Traversing graph for temporal chronology...")

for target_key in complaints_set:
    state, article_id = target_key

    # 1. Get the date of the FIRST complaint for this State-Article pair
    complaint_dates = []
    # complaints_map still refers to the one populated in 9bfcvB3FO8f0, which stores strings (doc_id)
    # This loop is correct for that structure.
    for doc_id in complaints_map[target_key]:
        doc_node = G.nodes.get(doc_id) # Use .get() for safer node access
        if doc_node:
            pub_date = doc_node.get("publication_date")
            if pd.notna(pub_date):
                try:
                    complaint_dates.append(pd.to_datetime(pub_date))
                except:
                    # Log or handle parsing errors for dates
                    pass

    if not complaint_dates:
        continue

    earliest_complaint = min(complaint_dates)

    # 2. Extract reporting verdicts that occurred BEFORE the first complaint
    for item in reporting_map.get(target_key, []):
        # Check if the item is a tuple of two elements before unpacking
        if isinstance(item, tuple) and len(item) == 2:
            verdict, report_doc_id = item
            doc_node = G.nodes.get(report_doc_id) # Use .get() for safer node access
            if doc_node:
                pub_date = doc_node.get("publication_date")

                if pd.notna(pub_date):
                    try:
                        report_date = pd.to_datetime(pub_date)
                        # Strict temporal check: Did the report happen before the complaint?
                        if report_date < earliest_complaint:
                            temporal_targeted_verdicts.append(verdict)
                    except:
                        failed_temporal_matches += 1
            else:
                # Node not found for report_doc_id
                failed_temporal_matches += 1
        else:
            # Increment a counter for malformed entries, indicating an issue in reporting_map population
            malformed_reporting_entries += 1
            # Optionally log the malformed item for debugging
            # print(f"Skipping malformed entry in reporting_map for {target_key}: {item}")

# 3. Analyze the temporally filtered data
if temporal_targeted_verdicts:
    temporal_df = pd.Series(temporal_targeted_verdicts).value_counts(normalize=True) * 100
    print("\n--- TEMPORALLY FILTERED DISTRIBUTION (Pre-Complaint Reports Only) ---")
    print(temporal_df.round(2))
else:
    print("No valid temporal matches found. Check date formatting or reporting_map content.")

if malformed_reporting_entries > 0:
    print(f"\nWarning: Encountered {malformed_reporting_entries} malformed entries in reporting_map. Ensure reporting_map is populated with (verdict, doc_id) tuples.")

Traversing graph for temporal chronology...

--- TEMPORALLY FILTERED DISTRIBUTION (Pre-Complaint Reports Only) ---
C     54.35
NC    26.23
D     19.43
Name: proportion, dtype: float64


In [33]:
# 7. Identifying Friction Points (Outlier Articles)
article_friction_map = defaultdict(lambda: {'Targeted_NC': 0, 'Total_Targeted': 0})

# Iterate through targeted pairs
for target_key in complaints_set:
    state, article_id = target_key

    # Get verdicts for this specific targeted pair
    # (Assuming reporting_map stores verdicts. Update if you changed it to tuples based on Cell 6)
    verdicts = reporting_map.get(target_key, [])

    just_verdicts = [v[0] for v in verdicts]
    nc_count = just_verdicts.count("NC")
    total_count = len(just_verdicts)

    if total_count > 0:
        article_friction_map[article_id]['Targeted_NC'] += nc_count
        article_friction_map[article_id]['Total_Targeted'] += total_count

# Calculate NC rates per article
friction_records = []
for art, counts in article_friction_map.items():
    if counts['Total_Targeted'] >= 10: # Minimum threshold to avoid noise
        nc_rate = (counts['Targeted_NC'] / counts['Total_Targeted']) * 100
        friction_records.append((art, counts['Total_Targeted'], nc_rate))

friction_df = pd.DataFrame(friction_records, columns=["Article", "Total Reports Evaluated", "NC Rate (%)"])
friction_df = friction_df.sort_values(by="NC Rate (%)", ascending=False).head(10)

print("--- TOP FRICTION POINTS (Articles driving the NC trend) ---")
display(friction_df)

--- TOP FRICTION POINTS (Articles driving the NC trend) ---


,Article,Total Reports Evaluated,NC Rate (%)
18,P2-04-04-163,54,92.592593
9,P2-17-01-163,22,77.272727
42,P2-31-02-163,12,75.000000
32,P2-03-02-163,11,72.727273
17,P2-31-03-163,14,71.428571
11,P2-31-01-163,15,60.000000
24,P2-02-00-163,113,59.292035
12,P2-15-03-163,13,53.846154
28,P2-24-00-163,18,50.000000
5,P2-06-04-163,143,49.650350


In [34]:
# 8. Intensity Analysis: Does NC Rate correlate with the Number of Complaints?
import scipy.stats as stats

intensity_records = []

print("Calculcating NC Rates and Complaint frequencies for all State-Article pairs...")

# Iterate over all pairs to capture the full spectrum (0 complaints to N complaints)
for target_key, verdicts_tuples in reporting_map.items():
    state, article_id = target_key

    # Calculate historical NC rate
    just_verdicts = [v[0] for v in verdicts_tuples]
    total_reports = len(just_verdicts)

    # We require a minimum number of reports (e.g., 5) to avoid statistical noise from new provisions
    if total_reports >= 5:
        nc_count = just_verdicts.count("NC")
        nc_rate = (nc_count / total_reports) * 100

        # Count total collective complaints for this exact pair
        complaint_count = len(complaints_map.get(target_key, []))

        intensity_records.append({
            "State": state,
            "Article": article_id,
            "Total_Reports": total_reports,
            "NC_Rate": nc_rate,
            "Complaint_Count": complaint_count
        })

intensity_df = pd.DataFrame(intensity_records)

# 1. Statistical Correlation Tests
pearson_corr, pearson_p = stats.pearsonr(intensity_df["NC_Rate"], intensity_df["Complaint_Count"])
spearman_corr, spearman_p = stats.spearmanr(intensity_df["NC_Rate"], intensity_df["Complaint_Count"])

print("\n--- STATISTICAL CORRELATION ---")
print(f"Spearman Correlation (Rank-based): {spearman_corr:.4f} (p-value: {spearman_p:.4e})")
print(f"Pearson Correlation (Linear): {pearson_corr:.4f} (p-value: {pearson_p:.4e})")

# 2. Dose-Response Analysis: Average complaints per severity bucket
# Grouping NC rates into buckets to see the trend clearly
bins = [-1, 0, 25, 50, 75, 100]
labels = ["0% NC", "1-25% NC", "26-50% NC", "51-75% NC", "76-100% NC"]
intensity_df["Severity_Bucket"] = pd.cut(intensity_df["NC_Rate"], bins=bins, labels=labels)

bucket_stats = intensity_df.groupby("Severity_Bucket")["Complaint_Count"].agg(['mean', 'max', 'count']).reset_index()
bucket_stats.rename(columns={'mean': 'Avg Complaints', 'max': 'Max Complaints', 'count': 'Number of Pairs'}, inplace=True)

print("\n--- DOSE-RESPONSE: COMPLAINTS BY NC SEVERITY BUCKET ---")
display(bucket_stats.round(3))

Calculcating NC Rates and Complaint frequencies for all State-Article pairs...

--- STATISTICAL CORRELATION ---
Spearman Correlation (Rank-based): 0.0383 (p-value: 8.7534e-02)
Pearson Correlation (Linear): 0.0086 (p-value: 7.0052e-01)

--- DOSE-RESPONSE: COMPLAINTS BY NC SEVERITY BUCKET ---


/tmp/ipykernel_2100/27087267.py:48: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket_stats = intensity_df.groupby("Severity_Bucket")["Complaint_Count"].agg(['mean', 'max', 'count']).reset_index()


,Severity_Bucket,Avg Complaints,Max Complaints,Number of Pairs
0,0% NC,0.066,6,731
1,1-25% NC,0.250,17,504
2,26-50% NC,0.209,10,374
3,51-75% NC,0.112,3,196
4,76-100% NC,0.102,4,187
